
# Week 11 – Vector Embeddings with ChromaDB

## Project Title
Creating and Storing Vector Embeddings using ChromaDB

## Objective
- Understand vector databases in LLM applications.
- Generate embeddings using Sentence Transformers.
- Store embeddings in ChromaDB.
- Perform semantic similarity searches.
- Retrieve the Top-5 relevant documents.
- Compare semantic search with keyword-based search.

## Technologies Used
- Python
- Sentence Transformers
- ChromaDB
- NumPy
- Pandas
- Scikit-learn

## Embedding Model
all-MiniLM-L6-v2



## 1. Import Required Libraries

The following libraries are used to load documents, generate embeddings, store vectors, and perform similarity searches.


In [1]:


import os
import csv
import numpy as np
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")


All libraries imported successfully!



## 2. Load the Sentence Transformer Model

The all-MiniLM-L6-v2 model converts text into numerical vector embeddings. Each embedding contains 384 dimensions.


In [2]:

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer model loaded successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer model loaded successfully!



## 3. Load Text Documents

In this step, we load all 10 AI-related text documents from the dataset/documents folder.

Each document contains information about Artificial Intelligence and its applications.


In [3]:

# Define the document folder path

DOCUMENT_FOLDER = "../dataset/documents"

# Create empty lists
documents = []
document_ids = []
metadatas = []

# Load all text documents
for filename in sorted(os.listdir(DOCUMENT_FOLDER)):

    if filename.endswith(".txt"):

        file_path = os.path.join(DOCUMENT_FOLDER, filename)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read().strip()

        if not content:
            continue

        # Extract title and category
        title = filename.replace(".txt", "")
        category = "Artificial Intelligence"

        for line in content.splitlines():

            if line.startswith("Title:"):
                title = line.replace("Title:", "").strip()

            elif line.startswith("Category:"):
                category = line.replace("Category:", "").strip()

        documents.append(content)
        document_ids.append(filename)

        metadatas.append({
            "title": title,
            "category": category,
            "source": filename
        })

print("Total documents loaded:", len(documents))
print("\nDocument Titles:")

for metadata in metadatas:
    print("-", metadata["title"])


Total documents loaded: 10

Document Titles:
- Introduction to Artificial Intelligence
- Artificial Intelligence Ethics
- Machine Learning
- Deep Learning
- Natural Language Processing
- Generative Artificial Intelligence
- Computer Vision
- Artificial Intelligence in Robotics
- Artificial Intelligence in Healthcare
- Artificial Intelligence in Education



## 4. Generate Vector Embeddings

The Sentence Transformer model all-MiniLM-L6-v2 converts each document into a numerical vector representation.

Each document is represented by a 384-dimensional embedding.


In [4]:

# Generate embeddings for all documents

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Display embedding shape
print("Embeddings generated successfully!")

print("Embedding shape:", embeddings.shape)

# Save embeddings
np.save("../dataset/embeddings.npy", embeddings)

print("Embeddings saved successfully!")


Embeddings generated successfully!
Embedding shape: (10, 384)
Embeddings saved successfully!



## 5. Create ChromaDB Collection

ChromaDB is a vector database used to store and retrieve document embeddings.

In this step, we create a persistent database and a collection named ai_documents_notebook.


In [5]:

# Create persistent ChromaDB client

client = chromadb.PersistentClient(
    path="../chroma_db"
)

# Create or access collection
collection = client.get_or_create_collection(
    name="ai_documents_notebook",
    metadata={
        "description": "AI document embeddings",
        "hnsw:space": "cosine"
    }
)

print("ChromaDB collection created successfully!")
print("Collection name:", collection.name)


ChromaDB collection created successfully!
Collection name: ai_documents_notebook



## 6. Store Documents and Metadata in ChromaDB

The generated embeddings, document contents, and metadata are stored in the ChromaDB collection.

Metadata includes document title, category, and source.


In [6]:

# Add documents to ChromaDB

if collection.count() == 0:

    collection.add(
        ids=document_ids,
        documents=documents,
        embeddings=embeddings.tolist(),
        metadatas=metadatas
    )

print("Documents stored successfully!")

print("Total documents in ChromaDB:", collection.count())


Documents stored successfully!
Total documents in ChromaDB: 10


## Step 16: Perform Semantic Similarity Search

Semantic search finds documents based on the meaning of a query rather than only matching exact words.

We use ChromaDB to retrieve the Top 5 most relevant documents for each query.

In [7]:
# Load the queries from queries.csv
queries_df = pd.read_csv("../dataset/queries.csv")

# Store all search results
notebook_results = []

# Perform semantic search for each query
for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    query_text = row["query"]

    # Generate embedding for the query
    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Retrieve the Top 5 relevant documents
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=5
    )

    print("\n" + "=" * 60)
    print("Query:", query_text)
    print("=" * 60)

    # Display retrieved documents
    for i in range(len(results["ids"][0])):

        document_id = results["ids"][0][i]
        metadata = results["metadatas"][0][i]
        document_text = results["documents"][0][i]
        distance = results["distances"][0][i]

        # Convert cosine distance into similarity
        similarity = 1 - distance

        print("\nRank:", i + 1)
        print("Document ID:", document_id)
        print("Title:", metadata["title"])
        print("Category:", metadata["category"])
        print("Similarity Score:", round(similarity, 4))

        notebook_results.append({
            "query_id": query_id,
            "query": query_text,
            "rank": i + 1,
            "document_id": document_id,
            "title": metadata["title"],
            "category": metadata["category"],
            "similarity_score": similarity,
            "document": document_text
        })

print("\nSemantic search completed successfully!")
print("Total results:", len(notebook_results))


Query: What is artificial intelligence and how does it work?

Rank: 1
Document ID: document1.txt
Title: Introduction to Artificial Intelligence
Category: Artificial Intelligence
Similarity Score: 0.7426

Rank: 2
Document ID: document7.txt
Title: Artificial Intelligence in Robotics
Category: Robotics
Similarity Score: 0.5958

Rank: 3
Document ID: document8.txt
Title: Artificial Intelligence in Healthcare
Category: Healthcare
Similarity Score: 0.5809

Rank: 4
Document ID: document9.txt
Title: Artificial Intelligence in Education
Category: Education
Similarity Score: 0.5715

Rank: 5
Document ID: document2.txt
Title: Machine Learning
Category: Machine Learning
Similarity Score: 0.5384

Query: How can computers learn from data?

Rank: 1
Document ID: document2.txt
Title: Machine Learning
Category: Machine Learning
Similarity Score: 0.4718

Rank: 2
Document ID: document1.txt
Title: Introduction to Artificial Intelligence
Category: Artificial Intelligence
Similarity Score: 0.3622

Rank: 3
Doc

## Step 17: Save Semantic Search Results

The retrieved documents and their similarity scores are saved into a CSV file for future analysis.

In [8]:
# Convert search results into a DataFrame
results_df = pd.DataFrame(notebook_results)

# Save results to CSV
results_df.to_csv(
    "../dataset/notebook_retrieved_results.csv",
    index=False
)

print("Semantic search results saved successfully!")
print("File: dataset/notebook_retrieved_results.csv")

# Display the first 10 results
results_df.head(10)

Semantic search results saved successfully!
File: dataset/notebook_retrieved_results.csv


,query_id,query,rank,document_id,title,category,similarity_score,document
0,Q1,What is artificial intelligence and how does i...,1,document1.txt,Introduction to Artificial Intelligence,Artificial Intelligence,0.742553,Title: Introduction to Artificial Intelligence...
1,Q1,What is artificial intelligence and how does i...,2,document7.txt,Artificial Intelligence in Robotics,Robotics,0.595849,Title: Artificial Intelligence in Robotics\nCa...
2,Q1,What is artificial intelligence and how does i...,3,document8.txt,Artificial Intelligence in Healthcare,Healthcare,0.580865,Title: Artificial Intelligence in Healthcare\n...
3,Q1,What is artificial intelligence and how does i...,4,document9.txt,Artificial Intelligence in Education,Education,0.571522,Title: Artificial Intelligence in Education\nC...
4,Q1,What is artificial intelligence and how does i...,5,document2.txt,Machine Learning,Machine Learning,0.538368,Title: Machine Learning\nCategory: Machine Lea...
5,Q2,How can computers learn from data?,1,document2.txt,Machine Learning,Machine Learning,0.471806,Title: Machine Learning\nCategory: Machine Lea...
6,Q2,How can computers learn from data?,2,document1.txt,Introduction to Artificial Intelligence,Artificial Intelligence,0.362164,Title: Introduction to Artificial Intelligence...
7,Q2,How can computers learn from data?,3,document6.txt,Computer Vision,Computer Vision,0.309804,Title: Computer Vision\nCategory: Computer Vis...
8,Q2,How can computers learn from data?,4,document9.txt,Artificial Intelligence in Education,Education,0.304563,Title: Artificial Intelligence in Education\nC...
9,Q2,How can computers learn from data?,5,document8.txt,Artificial Intelligence in Healthcare,Healthcare,0.299661,Title: Artificial Intelligence in Healthcare\n...


## Step 18: Keyword Search Comparison

Keyword search retrieves documents based on exact word matching.

In this step, we compare keyword search results with semantic search results to understand the difference between traditional search and AI-powered semantic search.

In [9]:

# Import required libraries
import os
import glob
import pandas as pd

# Load queries
queries_df = pd.read_csv("../dataset/queries.csv")

# Folder containing AI documents
documents_path = "../dataset/documents"

# Load all text documents
document_files = glob.glob(
    os.path.join(documents_path, "*.txt")
)

# Store keyword search results
keyword_results = []

# Perform keyword search for each query
for _, row in queries_df.iterrows():

    query_id = row["query_id"]
    query_text = row["query"]

    # Convert query into lowercase words
    query_words = set(query_text.lower().split())

    document_scores = []

    # Search through all documents
    for file_path in document_files:

        # Read document content
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        # Extract document information
        document_id = os.path.basename(file_path)
        title = os.path.splitext(document_id)[0]

        # Calculate keyword matching score
        document_words = set(content.lower().split())
        keyword_score = len(query_words.intersection(document_words))

        # Store document results
        document_scores.append({
            "query_id": query_id,
            "query": query_text,
            "document_id": document_id,
            "title": title,
            "category": "Artificial Intelligence",
            "keyword_score": keyword_score,
            "content": content
        })

    # Sort documents by keyword score
    document_scores.sort(
        key=lambda x: x["keyword_score"],
        reverse=True
    )

    # Retrieve Top 5 documents
    top_results = document_scores[:5]

    # Display results
    print("\n" + "=" * 60)
    print("Query:", query_text)
    print("=" * 60)

    for rank, result in enumerate(top_results, start=1):

        result["rank"] = rank
        keyword_results.append(result)

        print("\nRank:", rank)
        print("Document:", result["title"])
        print("Category:", result["category"])
        print("Keyword Score:", result["keyword_score"])

# Display total results
print("\nTotal keyword search results:", len(keyword_results))



Query: What is artificial intelligence and how does it work?

Rank: 1
Document: document2
Category: Artificial Intelligence
Keyword Score: 5

Rank: 2
Document: document4
Category: Artificial Intelligence
Keyword Score: 5

Rank: 3
Document: document5
Category: Artificial Intelligence
Keyword Score: 5

Rank: 4
Document: document6
Category: Artificial Intelligence
Keyword Score: 5

Rank: 5
Document: document1
Category: Artificial Intelligence
Keyword Score: 4

Query: How can computers learn from data?

Rank: 1
Document: document2
Category: Artificial Intelligence
Keyword Score: 3

Rank: 2
Document: document6
Category: Artificial Intelligence
Keyword Score: 2

Rank: 3
Document: document1
Category: Artificial Intelligence
Keyword Score: 1

Rank: 4
Document: document3
Category: Artificial Intelligence
Keyword Score: 1

Rank: 5
Document: document4
Category: Artificial Intelligence
Keyword Score: 1

Query: What are neural networks used for?

Rank: 1
Document: document3
Category: Artificial In

## Step 19: Compare Semantic Search and Keyword Search

In this step, we compare the results obtained from semantic search and keyword search.

Semantic search uses vector embeddings to understand the meaning of queries, whereas keyword search identifies exact matching words.

The comparison helps us understand the advantages of semantic search in AI applications.

In [15]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nDataset folder exists:")
print(os.path.exists("dataset"))

print("\nFiles inside dataset folder:")

if os.path.exists("dataset"):
    for file in os.listdir("dataset"):
        print(file)

Current Working Directory:
c:\Users\Admin\Desktop\Vector_Embeddings\notebook

Dataset folder exists:
False

Files inside dataset folder:


## Step 20: Save Search Comparison Results

The comparison results are saved into a CSV file for future analysis.

In [18]:
print("Comparison DataFrame exists:",
      "comparison_df" in globals())

Comparison DataFrame exists: False


In [ ]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nDataset folder exists:")
print(os.path.exists("dataset"))

print("\nFiles inside dataset folder:")

if os.path.exists("dataset"):
    for file in os.listdir("dataset"):
        print(file)